# Práctica — Fine-tuning de un LLM con LoRA

**NOMBRE:** Iván Alonso Vidal

En este notebook vamos a practicar cómo se adapta un LLM mediante **fine-tuning eficiente con LoRA**.

La idea principal no es entrenar un modelo desde cero, sino partir de un modelo ya preentrenado y adaptarlo entrenando solo una pequeña parte de sus parámetros.

Durante el notebook vamos a:

- cargar un modelo de lenguaje pequeño;
- entender cómo se formatea una conversación;
- ver cómo el modelo genera texto token a token;
- experimentar con parámetros de generación como `temperature` y `top_p`;
- aplicar fine-tuning eficiente con **LoRA**;
- comparar el comportamiento antes y después del fine-tuning;
- comparar **prompting vs fine-tuning**;
- diseñar un mini dataset propio de adaptación;
- decidir cuándo usar **Prompting**, **RAG** o **Fine-tuning**.

> Recomendación para Colab: activa GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.

# Instalación de librerías

Instalamos las librerías necesarias. La primera ejecución puede tardar unos minutos.

In [27]:
%pip install -q datasets --upgrade
%pip install -q transformers peft lion-pytorch mitdeeplearning matplotlib pandas
%pip install -q torchao==0.17.0

# Imports y configuración

Usaremos:

- `transformers` para cargar el modelo y tokenizador.
- `peft` para aplicar LoRA.
- `mitdeeplearning` para cargar datasets de estilo ya preparados.
- `torch` para el entrenamiento.

In [28]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader

import mitdeeplearning as mdl

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from lion_pytorch import Lion

pd.set_option("display.max_colwidth", 120)

In [29]:
if torch.backends.mps.is_available():
    print("MPS está disponible, pero usaremos CPU para mayor compatibilidad.")
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

Dispositivo: cuda


# Plantillas de chat

Un modelo conversacional no recibe solo una pregunta suelta. Normalmente espera una estructura que indique:

- cuándo habla el usuario;
- cuándo responde el modelo;
- dónde termina cada turno.

Esta estructura se conoce como **chat template** o plantilla de chat.

In [30]:
template_without_answer = "<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n"
template_with_answer = template_without_answer + "{answer}<end_of_turn>\n"

example_prompt = template_with_answer.format(
    question="What is your name?",
    answer="My name is Gemma!"
)

print(example_prompt)

<start_of_turn>user
What is your name?<end_of_turn>
<start_of_turn>model
My name is Gemma!<end_of_turn>



### TODO — Comprender la plantilla

Fíjate en la plantilla anterior y responde:

- ¿Qué parte marca el turno del usuario?

    La etiqueta "start_of_turn user\n" al inicio y "end_of_turn" al final de su intervención.
- ¿Qué parte marca el turno del modelo?

    La etiqueta "start_of_turn model\n", seguida de la respuesta y finalizando con "end_of_turn".
- ¿Qué problema podría aparecer si entrenas con una plantilla y luego usas otra distinta en inferencia?

    El modelo se confundiría. Al estar entrenado para predecir basándose en unos tokens estructurales concretos, si se los cambias en inferencia, el modelo podría no saber cuándo empezar a hablar, alucinar turnos, responder por el usuario o no detenerse nunca.

# Tokenización

Para que un LLM procese texto, primero hay que convertirlo en números.

La **tokenización** transforma texto en tokens numéricos. Después, el modelo trabaja con esos tokens.

También necesitamos el proceso inverso: convertir tokens de nuevo a texto.

In [31]:
model_id = "unsloth/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tamaño del vocabulario: {len(tokenizer.get_vocab())}")

Tamaño del vocabulario: 262145


In [32]:
text = "Here is some sample text!"

print("Texto original:")
print(text)

tokens = tokenizer.encode(text, return_tensors="pt")
print("\nTokens:")
print(tokens)

decoded_text = tokenizer.decode(tokens[0], skip_special_tokens=True)
print("\nTexto decodificado:")
print(decoded_text)

Texto original:
Here is some sample text!

Tokens:
tensor([[     2,   8291,    563,   1070,   5498,   1816, 236888]])

Texto decodificado:
Here is some sample text!


En la siguiente celda vas a probar cómo tokeniza el modelo diferentes tipos de texto.

No buscamos solo ejecutar el código: fíjate especialmente en palabras largas, palabras raras, símbolos y mezcla de idiomas.

In [33]:
# TODO: añade al menos 5 textos variados.
# Incluye:
# - una frase normal;
# - una palabra larga o rara;
# - una frase con números/símbolos;
# - una frase en español;
# - una frase relacionada con LLMs, RAG o fine-tuning.

texts_to_tokenize = [
    "This is a normal and simple sentence.",
    "Supercalifragilisticoespialidoso", # Palabra muy larga/rara
    "El ID de mi cuenta es: #993-X2, ¡gracias!", # Símbolos y números
    "Hola, ¿cómo estás hoy? Hace un día estupendo.", # Español
    "El fine-tuning eficiente con LoRA adapta los pesos del LLM sin requerir tanta VRAM." # Jerga técnica
]

for text in texts_to_tokenize:
    encoded = tokenizer.encode(text)
    decoded_tokens = [tokenizer.decode([token]) for token in encoded]

    print("=" * 100)
    print("Texto:", text)
    print("Número de tokens:", len(encoded))
    print("IDs:", encoded)
    print("Tokens decodificados:", decoded_tokens)

Texto: This is a normal and simple sentence.
Número de tokens: 9
IDs: [2, 2094, 563, 496, 3867, 532, 3606, 13315, 236761]
Tokens decodificados: ['<bos>', 'This', ' is', ' a', ' normal', ' and', ' simple', ' sentence', '.']
Texto: Supercalifragilisticoespialidoso
Número de tokens: 11
IDs: [2, 16769, 1653, 584, 16100, 543, 93437, 24132, 706, 6441, 236748]
Tokens decodificados: ['<bos>', 'Super', 'cal', 'if', 'rag', 'il', 'istico', 'esp', 'ial', 'idos', 'o']
Texto: El ID de mi cuenta es: #993-X2, ¡gracias!
Número de tokens: 20
IDs: [2, 4976, 5853, 569, 3628, 16674, 1115, 236787, 997, 236819, 236819, 236800, 236772, 236917, 236778, 236764, 21764, 1568, 23640, 236888]
Tokens decodificados: ['<bos>', 'El', ' ID', ' de', ' mi', ' cuenta', ' es', ':', ' #', '9', '9', '3', '-', 'X', '2', ',', ' ¡', 'gr', 'acias', '!']
Texto: Hola, ¿cómo estás hoy? Hace un día estupendo.
Número de tokens: 14
IDs: [2, 21529, 236764, 7196, 88565, 64135, 16229, 236881, 177448, 723, 12840, 217155, 4362, 236761]
Toke

### TODO — Reflexión sobre tokenización

Responde brevemente:

- ¿Qué texto se dividió en más tokens?
    La frase con jerga técnica
- ¿Qué ocurrió con la palabra larga o rara?

    Se dividió en múltiples fragmentos sin sentido aparente (sub-tokens) porque la palabra completa no existe en el diccionario fijo del tokenizador.
- ¿Por qué crees que los modelos modernos usan subpalabras en lugar de solo palabras completas?

    Para mantener un equilibrio ya que un vocabulario por letras requeriría secuencias inmensas (lento y poca memoria de contexto), y un vocabulario por palabras completas haría que el tamaño del diccionario fuera infinito (imposible procesar palabras nuevas o errores tipográficos). Las subpalabras solucionan ambos problemas.

# Cargar el modelo base

Cargamos el modelo base. Puede tardar unos minutos.

> Si Colab se queda sin memoria, reinicia el entorno, activa GPU y ejecuta de nuevo desde el inicio.

In [34]:
try:
    del model
    torch.cuda.empty_cache()
except NameError:
    pass

In [35]:
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("Modelo cargado en:", next(model.parameters()).device)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Modelo cargado en: cuda:0


# Generación token a token

Los LLMs no generan una respuesta completa de golpe.

Funcionan prediciendo el **siguiente token** dado un contexto. Después añaden ese token al texto y repiten el proceso.

In [36]:
question = "What is the capital of France? Use one word."
prompt = template_without_answer.format(question=question)

tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model(tokens)
    next_token_logits = output.logits[0, -1, :]

probs = torch.softmax(next_token_logits, dim=-1)

top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)

for prob, idx in zip(top_probs, top_indices):
    token_text = tokenizer.decode([idx.item()])
    print(f"{repr(token_text):>15} -> {prob.item():.4f}")

        'Paris' -> 1.0000
          'Par' -> 0.0000
            'P' -> 0.0000
           '巴黎' -> 0.0000
           'パリ' -> 0.0000
          'PAR' -> 0.0000
        'paris' -> 0.0000
           'Ly' -> 0.0000
           'Па' -> 0.0000
       ' Paris' -> 0.0000


### TODO — Interpretar la predicción del siguiente token

Responde:

- ¿El token más probable tiene sentido para el prompt?

    Sí, el modelo asigna la mayor probabilidad a la respuesta correcta (París)
- ¿Aparecen alternativas razonables?

    Sí, aparecen variaciones de mayúsculas como "paris".
- ¿Qué crees que ocurriría si el prompt fuese ambiguo?

    La distribución de probabilidades sería mucho más plana (valores más bajos repartidos entre muchas opciones), porque el modelo estaría menos "seguro" de cuál es la continuación correcta.

# Función de chat

Creamos una función `chat` para reutilizar el flujo completo:

```text
pregunta → plantilla → tokenización → generación → decodificación
```

In [37]:
def chat(question, max_new_tokens=80, temperature=0.7, top_p=0.5, only_answer=False):
    # TODO: prueba después a modificar max_new_tokens, temperature y top_p
    # para ver cómo afectan a la longitud, creatividad y estabilidad de la respuesta.
    prompt = template_without_answer.format(question=question)
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    if only_answer:
        if "<start_of_turn>model" in decoded:
            decoded = decoded.split("<start_of_turn>model")[-1]
        if "<end_of_turn>" in decoded:
            decoded = decoded.split("<end_of_turn>")[0]
        return decoded.strip()

    return decoded

In [38]:
print(chat(
    "Explain what NLP is in one sentence.",
    only_answer=True,
    max_new_tokens=80,
))

Natural Language Processing (NLP) is a field of computer science that focuses on enabling computers to understand, interpret, and generate human language.


# Experimentar con parámetros de generación

Ahora vas a observar cómo cambian las respuestas al modificar:

- `temperature`: controla la aleatoriedad.
- `top_p`: controla cuánta probabilidad acumulada se considera.

En esta parte el objetivo es experimentar, no encontrar una única respuesta correcta.

In [39]:
# TODO: añade 3 frases/prompts diferentes.
# Recomendación:
# - una tarea creativa;
# - una explicación técnica;
# - una respuesta con tono profesional o comercial.

generation_prompts = [
    "Escribe un poema breve de cuatro líneas sobre el espacio exterior.", # Creativo
    "Explica en un párrafo cómo funciona la gravedad usando términos científicos.", # Técnico
    "Redacta un correo electrónico formal para solicitar una reunión comercial de 15 minutos." # Profesional
]

# TODO: completa 3 configuraciones:
# - una conservadora;
# - una equilibrada;
# - una creativa.
generation_experiments = [
    {"temperature": 0.1, "top_p": 0.1, "label": "conservadora"},
    {"temperature": 0.7, "top_p": 0.7, "label": "equilibrada"},
    {"temperature": 1.5, "top_p": 0.95, "label": "creativa"},
]

for prompt in generation_prompts:
    print("\n" + "#" * 120)
    print("PROMPT:", prompt)

    for config in generation_experiments:
        print("\n" + "=" * 100)
        print("Configuración:", config["label"], config)
        print(chat(
            prompt,
            only_answer=True,
            max_new_tokens=100,
            temperature=config["temperature"],
            top_p=config["top_p"],
        ))


########################################################################################################################
PROMPT: Escribe un poema breve de cuatro líneas sobre el espacio exterior.

Configuración: conservadora {'temperature': 0.1, 'top_p': 0.1, 'label': 'conservadora'}
Aquí tienes un poema breve de cuatro líneas sobre el espacio exterior:

En la oscuridad, las estrellas brillan,
Un lienzo infinito, sin fin ni límites.
El cosmos se extiende, un misterio profundo,
Un viaje sin fin, para el alma que se encuentra.

Configuración: equilibrada {'temperature': 0.7, 'top_p': 0.7, 'label': 'equilibrada'}
Aquí tienes un poema breve de cuatro líneas sobre el espacio exterior:

En vastos campos de negro y luz,
Estrellas brillan, un eterno resplandor.
El cosmos se extiende, sin fin, sin duda,
Un misterio profundo, para siempre.

Configuración: creativa {'temperature': 1.5, 'top_p': 0.95, 'label': 'creativa'}
A pesar de estar a toda hora cerca,
A la oscuridad el universo observa con 

### TODO — Análisis de los parámetros de generación

Responde después de ejecutar varios prompts:

- ¿Qué diferencias viste entre la configuración conservadora, equilibrada y creativa?

    La conservadora es directa, predecible y puede volverse repetitiva. La equilibrada es natural. La creativa usa vocabulario más florido, da giros inesperados y, si es muy alta, empieza a perder cohesión (alucina palabras o pierde el hilo).
- ¿En qué tipo de prompt se notó más el cambio?

    En el prompt creativo (el poema).
- ¿Qué configuración usarías para un asistente de soporte técnico? ¿Por qué?

    Una conservadora (temperature baja, ~0.1-0.3). Queremos respuestas precisas, deterministas y basadas en hechos, sin espacio para "inventiva".
- ¿Qué configuración usarías para generar ideas creativas? ¿Por qué?

    Una creativa (temperature ~0.8-1.2, top_p alto). Queremos diversidad, salir de los patrones comunes y generar múltiples opciones distintas ante el mismo prompt.
- ¿Viste algún caso donde una configuración más creativa empeorase la respuesta?

    Sí, al pedir una explicación técnica con creatividad alta, el modelo divaga y no llega a responder del todo la duda.

# Dataset de instrucciones y respuestas - Estilo **Leprechaun**

Ahora cargaremos un dataset de instrucciones y respuestas.

Para esta práctica, usaremos una versión preparada con respuestas en un estilo concreto.

Primero trabajaremos intentando que el LLM responda como un **leprechaun**.

In [40]:
ds = load_dataset("databricks/databricks-dolly-15k", split="train")
ds[0]

{'instruction': 'When did Virgin Australia start operating?',
 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.',
 'category': 'closed_qa'}

In [41]:
train_loader, test_loader = mdl.lab3.create_dataloader(style="leprechaun")

sample = train_loader.dataset[44]
question = sample["instruction"]
answer = sample["response"]
answer_style = sample["response_style"]

print(
    f"Pregunta: {question}\n\n"
    f"Respuesta original: {answer}\n\n"
    f"Respuesta con estilo: {answer_style}"
)

Pregunta: Are lilies safe for cats?

Respuesta original: No, lilies are toxic to cats if consumed and should not be kept in a household with cats

Respuesta con estilo: Och, no indeed, me hearty! Them lilies there be as dangerous as a pot o' gold guarded by a banshee to a wee kitty cat! If a whiskered lad or lass takes a bite of one, it's as bad as swallowing a curse from the old Hag herself. So, ye best keep them far from yer feline friends, or else ye'll be needin' more than just a four-leaf clover to bring luck back into yer home!


### TODO — Analizar el dataset

Responde:

- ¿Qué cambia entre `response` y `response_style`?

    "response" es una respuesta estándar de un asistente neutral. "response_style" tiene el mismo fondo semántico, pero está reescrita con jerga irlandesa exagerada, referencias a oro, arcoíris, tréboles y una sintaxis folclórica.
- ¿El cambio es solo de vocabulario o también de tono/estructura?

    Cambia todo: vocabulario (usando "hearty", "pot of gold"), el tono (más jovial y teatral) y la estructura de las frases.
- ¿Qué podría aprender el modelo de este dataset?

    A asumir el rol de un duende irlandés en todas sus respuestas, manteniendo su capacidad original de resolver dudas, pero cambiándoles el envoltorio.
- ¿Qué problemas podría provocar un dataset de estilo mal diseñado?

    Si las respuestas objetivo tienen mala gramática, son tóxicas, o pierden la respuesta original (ej. si al preguntar cuánto es 2+2 solo responde "¡Tengo oro!"), el modelo se volvería inútil.

# Fine-tuning eficiente con LoRA

Hacer fine-tuning completo de un LLM puede ser caro porque implica actualizar muchos pesos.

**LoRA** permite adaptar el modelo entrenando solo una pequeña parte:

- congela el modelo base;
- añade pequeñas matrices entrenables;
- reduce memoria y coste;
- facilita crear adaptadores distintos para tareas distintas.

In [42]:
def apply_lora(model):
    if isinstance(model, PeftModel):
        return model

    lora_config = LoraConfig(
        r=8,
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"
        ],
    )

    lora_model = get_peft_model(model, lora_config)
    return lora_model

model = apply_lora(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros totales:     {total_params:,}")
print(f"Porcentaje entrenable:  {100 * trainable_params / total_params:.4f}%")

Parámetros entrenables: 6,070,272
Parámetros totales:     1,005,956,224
Porcentaje entrenable:  0.6034%


### TODO — Interpretar LoRA

Responde:

- ¿Estamos entrenando todo el modelo?

    No. Con LoRA, los miles de millones de parámetros originales se congelan. Solo entrenamos unas matrices pequeñas inyectadas en las capas de atención.
- ¿Por qué LoRA reduce el coste?

    Porque al haber menos parámetros entrenables, se requieren muchos menos cálculos matemáticos (gradientes) y se ahorra drásticamente el consumo de memoria en la tarjeta gráfica (VRAM).
- ¿Qué ventaja tendría tener varios adaptadores LoRA sobre un mismo modelo base?

    Eficiencia y modularidad. Puedes tener un solo modelo pesado cargado en memoria, y cargar "al vuelo" pequeños adaptadores de pocos megabytes dependiendo del usuario: uno para que sea leprechaun, otro para código, otro para JSON, etc.

# Forward pass y cálculo de pérdida

Durante el entrenamiento, el modelo predice el siguiente token.

Comparamos esas predicciones con los tokens reales y calculamos una pérdida. Solo calcularemos la pérdida sobre la respuesta, porque queremos enseñar al modelo cómo debe responder.

In [43]:
def forward_and_compute_loss(model, tokens, mask, context_length=256):
    tokens = tokens[:, :context_length]
    mask = mask[:, :context_length]

    x = tokens[:, :-1]
    y = tokens[:, 1:]
    mask = mask[:, 1:]

    logits = model(x).logits

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1),
        reduction="none",
    )

    loss = loss[mask.reshape(-1)].mean()

    return loss

# Bucle de entrenamiento

Entrenaremos pocos pasos para que sea viable en clase.

**Importante:** más pasos no siempre significa mejor resultado. Con pocos datos, entrenar demasiado puede hacer que el modelo exagere el estilo o pierda utilidad.

In [44]:
def train(model, dataloader, tokenizer, max_steps=45, context_length=512, learning_rate=1e-3):
    # TODO: si tienes tiempo, experimenta con max_steps y learning_rate.
    # Pocos pasos pueden no adaptar suficiente; demasiados pueden exagerar el estilo.
    losses = []

    model = apply_lora(model)
    optimizer = Lion(model.parameters(), lr=learning_rate)

    for step, batch in enumerate(dataloader):
        question = batch["instruction"][0]
        answer = batch["response_style"][0]

        text = template_with_answer.format(question=question, answer=answer)

        ids = tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
        answer_start = text.index(answer)
        mask = ids["offset_mapping"][:, :, 0] >= answer_start

        loss = forward_and_compute_loss(
            model=model,
            tokens=ids["input_ids"],
            mask=mask,
            context_length=context_length,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if step % 5 == 0:
            print("=" * 80)
            print("Pregunta:", "What is your favorite color and why?")
            print("Respuesta:")
            print(chat("What is your favorite color and why?", only_answer=True, max_new_tokens=150)) # TODO: Imprime la respuesta del modelo a una pregunta que se te ocurra. Usa la función chat() creada anteriormente.
            print(f"step {step} | loss media: {torch.mean(torch.tensor(losses)).item():.4f}")
            losses = []

        if step > 0 and step % max_steps == 0:
            break

    return model

In [45]:
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("Modelo cargado en:", next(model.parameters()).device)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Modelo cargado en: cuda:0


In [46]:
model = train(model, train_loader, tokenizer, max_steps=45)

Pregunta: What is your favorite color and why?
Respuesta:
As a large language model, I don’t *experience* colors in the same way a human does. I don’t have eyes to see them! However, if I were to choose a "favorite" based on the data I’ve processed, I’d say **blue**. 

Here’s why, based on the vast amount of text and images I’ve analyzed:

* **It’s often associated with calmness and tranquility:** Blue is frequently linked to peace, serenity, and stability. It’s a color that promotes relaxation and a sense of well-being.
* **It’s used in so many different contexts:** From the vastness of the ocean to the clear sky, blue appears repeatedly in
step 0 | loss media: 3.0312
Pregunta: What is your favorite color and why?
Respuesta:
Top o' the mornin' to ye, me hearty! As for me favorite color, that's a mighty fine question, and one that's been on me mind a bit. I reckon it's a deep, rich, earthy **forest green**. It reminds me of the deep, cool shade of a forest canopy, full of moss and fern

# Comparar el modelo tras el fine-tuning

Probamos varias preguntas y observamos si el estilo entrenado aparece en la respuesta.

In [47]:
# TODO: añade 5 prompts variados para comprobar si el estilo aparece.
# Incluye preguntas informativas, creativas y de consejo.

test_prompts = [
    "What do you think about humans?",
    "How do you usually travel from one place to another?",
    "Tell me a secret about the universe.",
    "What is your favorite color and why?"
]

for prompt in test_prompts:
    print("=" * 100)
    print("Pregunta:", prompt)
    print("Respuesta:")
    print(chat(prompt, only_answer=True, max_new_tokens=150))

Pregunta: What do you think about humans?
Respuesta:
Top o' the mornin' to ye now! Ye ask about humans? Well, let me tell ye what. They're a bit o' a mess, wouldn't ye agree? They're full o' both good and bad, like a storm before the rain. They're full o' kindness and compassion, but they're also full o' greed and selfishness. They're full o' love and loyalty, but they're also full o' jealousy and envy. They're full o' ambition and drive, but they're also full o' doubt and fear.

Now, that's a bit o' a stretch, wouldn't it? But hey, at least it's
Pregunta: How do you usually travel from one place to another?
Respuesta:
Ah, top o' the mornin' to ye! Now, ye ask about travel? Well, let me tell ye. First, ye'll need to pack yer bags, ye hear? And then ye'll need to find yer trusty map, or perhaps ye'll use the internet, which is as reliable as a leprechaun's pot o' gold. Next, ye'll need to find yer transportation, which could be a car, train, bus, or plane. And lastly, ye'll need to find

### TODO — Comparar antes y después del fine-tuning

Responde:

- ¿Notas un cambio de estilo respecto al modelo base?

    Sí, el modelo empieza a responder de forma evidente como un duende.
- ¿El modelo mantiene la utilidad de la respuesta?

    Generalmente sí, sigue respondiendo a la duda planteada pero adaptada al estilo.
- ¿El estilo aparece siempre o solo a veces?

    Debería aparecer casi siempre, pero a veces, si la pregunta es muy técnica o compleja, el modelo puede "olvidar" el estilo a mitad de respuesta y volver a su forma neutral.
- ¿Qué crees que pasaría si entrenamos muchos más pasos?

    Ocurriría un sobreajuste (overfitting). El modelo exageraría tanto el estilo que podría dejar de responder a la pregunta real o generar balbuceos incomprensibles.

# Segunda adaptación: estilo Yoda

Ahora cargaremos un segundo estilo.

Esta parte sirve para observar qué ocurre cuando seguimos adaptando el mismo modelo con otro estilo.

In [48]:
train_loader, test_loader = mdl.lab3.create_dataloader(style="yoda")

sample = train_loader.dataset[10]
print("Pregunta:")
print(sample["instruction"])
print("\nRespuesta original:")
print(sample["response"])
print("\nRespuesta estilo Yoda:")
print(sample["response_style"])

Map (num_proc=1):   0%|          | 0/2048 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=21270) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Pregunta:
Who is Thomas Jefferson?

Respuesta original:
Thomas Jefferson (April 13, 1743 – July 4, 1826) was an American statesman, diplomat, lawyer, architect, philosopher, and Founding Father who served as the third president of the United States from 1801 to 1809. Among the Committee of Five charged by the Second Continental Congress with authoring the Declaration of Independence, Jefferson was the Declaration's primary author. Following the American Revolutionary War and prior to becoming the nation's third president in 1801, Jefferson was the first United States secretary of state under George Washington and then the nation's second vice president under John Adams.

Among the nation's Founding Fathers, Jefferson is considered unmatched in his intellectual depth and breadth. His passionate writings and advocacy for human rights, including freedom of thought, speech, and religion, were a leading inspiration behind the American Revolution, which ultimately gave rise to the American R

### TODO — Hipótesis antes del segundo entrenamiento

Antes de entrenar, responde:

- ¿Qué rasgos del estilo Yoda esperas que aprenda?

    Su sintaxis clásica de invertir el orden de la oración (Objeto-Sujeto-Verbo) y un tono sabio/pausado.
- ¿Crees que puede olvidar parte del estilo anterior?

    Totalmente. Esto se llama "olvido catastrófico" (catastrophic forgetting). Al actualizar los pesos de LoRA para el nuevo estilo, los patrones del leprechaun se sobrescribirán y se perderán.
- ¿Podría conseguirse algo parecido solo con prompting?

    Sí, diciendo "Actúa como Yoda" en el prompt base (System Prompt).

In [50]:
model = train(model, train_loader, tokenizer, max_steps=45)

Pregunta: What is your favorite color and why?
Respuesta:
Och, the color I've got is a deep, rich, crimson, as the blood that flows through the veins of a mighty tree. It's a color that evokes the heart of the earth, the very soul of the world. And why, it's a color that reminds me of the love that I have for you, my dear.
step 0 | loss media: 2.9688
Pregunta: What is your favorite color and why?
Respuesta:
Red, it is. Red, the color of love, passion, and desire. Red, the color of life, energy, and vitality. Red, the color of the heart, the color of love, the color of life, the color of the soul. Red, the color of the rainbow, the color of hope, the color of dreams, the color of happiness, the color of joy, the color of life, the color of joy, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life, the color of life
step 5 | loss media

In [51]:
for prompt in test_prompts:
    print("=" * 100)
    print("Pregunta:", prompt)
    print("Respuesta:")
    print(chat(prompt, only_answer=True, max_new_tokens=150))

Pregunta: What do you think about humans?
Respuesta:
Incomprehensibility, the most challenging aspect of humanity is.
Pregunta: How do you usually travel from one place to another?
Respuesta:
Via train, bus, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking
Pregunta: Tell me a secret about the universe.
Respuesta:
Via microscope, the universe is. Hmmm. Hmmm. Hmmm. Hmmm. Hmmm. Hmmm. Hmmm. Hmm

# Prompting vs fine-tuning

Ahora comparamos dos formas de conseguir un estilo:

- **Prompting:** pedir explícitamente el estilo en el prompt.
- **Fine-tuning:** adaptar el modelo para que tienda a responder con ese estilo.

El objetivo es razonar sobre ventajas y limitaciones, no obtener una métrica exacta.

In [52]:
# TODO: añade al menos 4 preguntas para comparar fine-tuning vs prompting.
comparison_questions = [
    "What do you think about humans?",
    "How do you usually travel from one place to another?",
    "Tell me a secret about the universe.",
    "What is your favorite color and why?"
]

for question in comparison_questions:
    prompted_question = f"Answer like Yoda: {question}"

    print("=" * 100)
    print("Pregunta normal:")
    print(question)
    print("\nRespuesta modelo fine-tuneado:")
    print(chat(question, only_answer=True, max_new_tokens=120))

    print("\nPregunta con prompting explícito:")
    print(prompted_question)
    print("\nRespuesta con prompting:")
    print(chat(prompted_question, only_answer=True, max_new_tokens=120))

Pregunta normal:
What do you think about humans?

Respuesta modelo fine-tuneado:
Complex, the human mind is. Interesting, it is. Complex, it is. Interesting, it is.

Complex, the human mind is.

Complex, the way we make it is.

Complex, the way we seek it is.

Complex, the way we make it is.

Complex, the way we seek it is.

Complex, the way we are.

Complex, the way we are.

Complex, the way we make it is.

Complex, the way we are.

Complex, the way we seek it is.

Complex, the way we

Pregunta con prompting explícito:
Answer like Yoda: What do you think about humans?

Respuesta con prompting:
Human, humans are. Interesting, this one is.
Pregunta normal:
How do you usually travel from one place to another?

Respuesta modelo fine-tuneado:
Via train, bus, walking, cycling, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, walking, w

### TODO — Comparar prompting y fine-tuning

Responde:

- ¿Qué opción consigue mejor el estilo?

    El fine-tuning suele conseguir un estilo más arraigado, natural y consistente a lo largo de interacciones largas, aunque un buen prompting también es de utilidad.
- ¿Cuál mantiene mejor la utilidad de la respuesta?

    A veces el prompting mantiene mejor los hechos porque no altera los pesos del modelo, mientras que un fine-tuning pobre puede degradar la calidad de la respuesta base.
- ¿Cuál sería más fácil de cambiar mañana si queremos otro estilo?

    El prompting, sin lugar a dudas. Solo cambias el texto del prompt. Cambiar un fine-tuning requiere volver a preparar un dataset y entrenar de nuevo.
- ¿Cuál tendría más sentido si queremos millones de respuestas con un formato estable?

    El fine-tuning. Te ahorra inyectar el enorme "System prompt" en cada petición (ahorrando tokens de entrada, lo que es igual a dinero en APIs) y fuerza al modelo a tener un comportamiento estructural por defecto.

# Reto libre guiado — Crear tu propia adaptación

En esta parte vas a crear una pequeña adaptación propia. La idea es poner en práctica el flujo completo:

```text
elegir comportamiento → crear ejemplos → formatear dataset → entrenar con LoRA → probar respuestas
```

**Objetivo:** que el modelo aprenda un comportamiento más concreto que un simple estilo.

Puedes elegir una de estas ideas o inventar otra:

- responder siempre en JSON;
- clasificar tickets;
- responder como tutor de FP;
- responder como soporte técnico;
- dar respuestas muy breves y profesionales;
- adaptar el tono a una marca.

In [53]:
# TODO: define una nueva tarea de adaptación.
# Elige un comportamiento concreto y crea al menos 8 ejemplos.
# Mantén las claves: instruction (esta debe tener el mismo valor en todoslos ejemplos), input, output.

custom_task_name = "Clasificador de Sentimientos en JSON"

custom_training_examples = [
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "Me encanta este producto, es absolutamente maravilloso y lo recomiendo a todos.", "output": '{"sentimiento": "positivo"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "Es la peor experiencia de mi vida, el servicio al cliente fue pésimo.", "output": '{"sentimiento": "negativo"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "El paquete llegó hoy. La caja es azul y pesa dos kilos.", "output": '{"sentimiento": "neutral"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "¡Fantástico trabajo! El equipo ha superado todas las expectativas este trimestre.", "output": '{"sentimiento": "positivo"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "No sé qué pensar, la película estuvo pasable pero el final no me convenció.", "output": '{"sentimiento": "neutral"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "El envío se retrasó tres semanas y además el artículo llegó roto.", "output": '{"sentimiento": "negativo"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "Estoy muy feliz y agradecido por esta gran oportunidad.", "output": '{"sentimiento": "positivo"}'},
    {"instruction": "Clasifica el sentimiento del siguiente texto devolviendo un JSON.", "input": "La aplicación falla constantemente, me rindo con esto.", "output": '{"sentimiento": "negativo"}'},
]

pd.DataFrame(custom_training_examples)

,instruction,input,output
0,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,"Me encanta este producto, es absolutamente maravilloso y lo recomiendo a todos.","{""sentimiento"": ""positivo""}"
1,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,"Es la peor experiencia de mi vida, el servicio al cliente fue pésimo.","{""sentimiento"": ""negativo""}"
2,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,El paquete llegó hoy. La caja es azul y pesa dos kilos.,"{""sentimiento"": ""neutral""}"
3,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,¡Fantástico trabajo! El equipo ha superado todas las expectativas este trimestre.,"{""sentimiento"": ""positivo""}"
4,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,"No sé qué pensar, la película estuvo pasable pero el final no me convenció.","{""sentimiento"": ""neutral""}"
5,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,El envío se retrasó tres semanas y además el artículo llegó roto.,"{""sentimiento"": ""negativo""}"
6,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,Estoy muy feliz y agradecido por esta gran oportunidad.,"{""sentimiento"": ""positivo""}"
7,Clasifica el sentimiento del siguiente texto devolviendo un JSON.,"La aplicación falla constantemente, me rindo con esto.","{""sentimiento"": ""negativo""}"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### TODO — Revisión del diseño

Antes de entrenar, revisa tu dataset:

- ¿La tarea está claramente definida? Sí
- ¿Las salidas tienen un formato consistente? Sí, formato json.
- ¿Hay suficientes ejemplos variados? Sí, están equilibrados.
- ¿Incluiste algún caso difícil o ambiguo? Sí, uno neutral.

In [54]:
# Convertimos los ejemplos al formato que espera la función train:
# instruction: pregunta completa para el modelo (incluye instruction e input)
# response_style: respuesta objetivo

def build_instruction_from_custom_example(example):

    # TODO: crear instruction. completa de entrada
    instruction = f"{example['instruction']}\nTexto: {example['input']}"

    return instruction

custom_rows = [
    {
        "instruction": build_instruction_from_custom_example(example),
        "response_style": example["output"],
    }
    for example in custom_training_examples
]

custom_df = pd.DataFrame(custom_rows)
custom_df

,instruction,response_style
0,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Me encanta este producto, es absolutamente...","{""sentimiento"": ""positivo""}"
1,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Es la peor experiencia de mi vida, el serv...","{""sentimiento"": ""negativo""}"
2,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: El paquete llegó hoy. La caja es azul y pe...,"{""sentimiento"": ""neutral""}"
3,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: ¡Fantástico trabajo! El equipo ha superado...,"{""sentimiento"": ""positivo""}"
4,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: No sé qué pensar, la película estuvo pasab...","{""sentimiento"": ""neutral""}"
5,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: El envío se retrasó tres semanas y además ...,"{""sentimiento"": ""negativo""}"
6,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Estoy muy feliz y agradecido por esta gran...,"{""sentimiento"": ""positivo""}"
7,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: La aplicación falla constantemente, me rin...","{""sentimiento"": ""negativo""}"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [55]:
# TODO: revisa la longitud en tokens de tus ejemplos.
# Si algún ejemplo es demasiado largo, piensa cómo lo acortarías.

custom_df["formatted_text"] = custom_df.apply(
    lambda row: template_with_answer.format(
        question=row["instruction"],
        answer=row["response_style"]
    ),
    axis=1,
)

custom_df["token_length"] = custom_df["formatted_text"].apply(
    lambda text: len(tokenizer.encode(text))
)

custom_df[["instruction", "response_style", "token_length"]]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,instruction,response_style,token_length
0,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Me encanta este producto, es absolutamente...","{""sentimiento"": ""positivo""}",48
1,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Es la peor experiencia de mi vida, el serv...","{""sentimiento"": ""negativo""}",51
2,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: El paquete llegó hoy. La caja es azul y pe...,"{""sentimiento"": ""neutral""}",48
3,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: ¡Fantástico trabajo! El equipo ha superado...,"{""sentimiento"": ""positivo""}",50
4,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: No sé qué pensar, la película estuvo pasab...","{""sentimiento"": ""neutral""}",52
5,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: El envío se retrasó tres semanas y además ...,"{""sentimiento"": ""negativo""}",49
6,Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Estoy muy feliz y agradecido por esta gran...,"{""sentimiento"": ""positivo""}",45
7,"Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: La aplicación falla constantemente, me rin...","{""sentimiento"": ""negativo""}",46


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [57]:
# TODO: crea un DataLoader y entrena algunos pasos.
# Recomendación: empieza con max_steps entre 15 y 30 para no alargar demasiado la práctica.

custom_dataset = Dataset.from_pandas(custom_df[["instruction", "response_style"]])
custom_loader = DataLoader(custom_dataset, batch_size=1, shuffle=True)

# Entrenamos el modelo por 25 pasos (suficiente para que pille la idea sin pasarse)
model = train(model, custom_loader, tokenizer, max_steps=25)

Pregunta: What is your favorite color and why?
Respuesta:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Blue, green, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple, purple,
step 0 | loss media: 4.2812
Pregunta: What is your favorite color and why?
Respuesta:
color: blue, green, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yellow, orange, purple, brown, yel

In [58]:
# TODO: crea al menos 4 prompts nuevos para probar si tu adaptación funcionó.
# Deben ser casos no idénticos a los ejemplos de entrenamiento.

custom_test_prompts = [
    "Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Hace un día precioso y tengo muchas ganas de salir a pasear.",
    "Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: Odio cuando el tren llega tarde, me arruina toda la mañana.",
    "Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: El informe tiene 15 páginas y se publicó el martes.",
    "Clasifica el sentimiento del siguiente texto devolviendo un JSON.\nTexto: ¡Qué tecnología tan increíble, estoy asombrado!"
]

for prompt in custom_test_prompts:
    print("=" * 100)
    print("Prompt:")
    print(prompt)
    print("\nRespuesta:")
    print(chat(prompt, only_answer=True, max_new_tokens=80, temperature=0.2, top_p=0.7))

Prompt:
Clasifica el sentimiento del siguiente texto devolviendo un JSON.
Texto: Hace un día precioso y tengo muchas ganas de salir a pasear.

Respuesta:
{"sentimiento": "positivo"}
Prompt:
Clasifica el sentimiento del siguiente texto devolviendo un JSON.
Texto: Odio cuando el tren llega tarde, me arruina toda la mañana.

Respuesta:
{"sentimiento": "positivo"}
Prompt:
Clasifica el sentimiento del siguiente texto devolviendo un JSON.
Texto: El informe tiene 15 páginas y se publicó el martes.

Respuesta:
{"sentimiento": "positivo"}
Prompt:
Clasifica el sentimiento del siguiente texto devolviendo un JSON.
Texto: ¡Qué tecnología tan increíble, estoy asombrado!

Respuesta:
{"sentimiento": "positivo"}


### TODO — Reflexión del reto libre

Responde:

- ¿Qué comportamiento intentaste enseñar?

    A clasificar sentimientos de texto y forzar que la salida fuera estrictamente un objeto JSON válido, eliminando la "charla" introductoria típica del asistente (ej: "Aquí tienes la clasificación:...").
- ¿El modelo empezó a seguir ese comportamiento?

    Sí, va directo al formato JSON.
- ¿Qué parte falló o salió inestable?

    No acaba de pillar bien los sentimientos, necesitaría más ejemplos para obtener mejores resultados.
- ¿Usarías fine-tuning, prompting o RAG para este caso en un proyecto real? ¿Por qué?

    Para forzar formatos de salida estrictos (como JSON) o tareas de clasificación muy repetitivas, el fine-tuning es ideal. Ahorra tokens y mejora el determinismo. RAG no aporta nada aquí porque no buscamos información externa.
- ¿Qué añadirías al dataset para mejorar la adaptación?

    Ejemplos con casos límite (textos vacíos, sarcasmo) y ejemplos negativos donde el sentimiento sea muy complejo, para que aprenda a manejar la incertidumbre (ej. devolver {"sentimiento": "desconocido"}).

# Reflexión final

Responde brevemente:

- ¿Qué diferencia hay entre adaptar el comportamiento del modelo y darle información externa?

    Adaptar comportamiento (Fine-tuning) cambia "cómo" habla y cómo estructura la información. Darle información (RAG) cambia "qué" sabe, permitiéndole buscar documentos recientes antes de responder.
- ¿Por qué fine-tuning no suele ser la mejor opción para documentos que cambian a menudo?

    Porque los LLMs son malos reteniendo hechos precisos durante el entrenamiento (tienden a alucinar), y si el documento cambia mañana, tendrías que volver a entrenar el modelo, lo cual es lento y costoso.
- ¿Qué papel tiene LoRA en el fine-tuning eficiente?

    Democratiza el entrenamiento. Permite adaptar modelos masivos usando hardware comercial accesible al no tocar todos los pesos de la red neuronal.
- ¿Por qué el dataset es tan importante?

    Porque rige la regla del Garbage In, Garbage Out. El modelo no sabe qué está bien o mal, solo imita los patrones de los datos que le das. Si el dataset es malo, el modelo será inútil.

# Conclusión

En esta práctica hemos recorrido un pipeline completo:

```text
modelo base → plantilla de chat → generación → LoRA → fine-tuning → comparación
```

Ideas clave:

- Un LLM genera texto token a token.
- Los parámetros de generación afectan a creatividad y estabilidad.
- LoRA permite hacer fine-tuning entrenando pocos parámetros.
- Prompting, RAG y fine-tuning sirven para problemas diferentes.
- Fine-tuning tiene sentido cuando queremos aprender patrones estables con ejemplos de calidad.
- Para información cambiante o documentos internos, normalmente conviene RAG.
- La calidad del dataset es crítica.